## **Model Evaluation**

#### **Loading Libraries**

In [14]:
import warnings
import pandas as pd 
import numpy as np 
import seaborn as sns  
from sklearn import tree  
import matplotlib.pyplot as plt     
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import  LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.metrics import mean_squared_error, r2_score, roc_curve, accuracy_score, classification_report, roc_auc_score,ConfusionMatrixDisplay, confusion_matrix
warnings.filterwarnings("ignore")

#### **Loading Data**

In [3]:
df = pd.read_csv("f1_strategy_dataset_v4.csv")
df.shape

(101371, 16)

In [4]:
df.dtypes

Driver                        str
LapNumber                   int64
Compound                      str
Stint                       int64
TyreLife                  float64
Position                    int64
LapTime (s)               float64
Race                          str
Year                        int64
LapTime_Delta             float64
Cumulative_Degradation    float64
PitStop                     int64
PitNextLap                  int64
RaceProgress              float64
Normalized_TyreLife       float64
Position_Change           float64
dtype: object

#### **Creating our pipeline**

In [7]:
categorical_cols = ['Driver','Compound', 'Race']

numerical_cols = ['LapNumber',  'Stint', 'TyreLife', 'Position',
       'LapTime (s)',  'Year', 'LapTime_Delta',
       'Cumulative_Degradation', 'PitStop',  'RaceProgress',
       'Normalized_TyreLife', 'Position_Change']

In [8]:
preprocessor = ColumnTransformer([
    ('num',StandardScaler(), numerical_cols), 
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols)
])

# pipeline 
#  Decision Tree
pipeline_dt = Pipeline(
    [
        ('prep', preprocessor), 
        ('clf', tree.DecisionTreeClassifier (class_weight='balanced', max_depth = 15))
    ]
)
#  Logistic regression
pipeline_lr = Pipeline(
    [
        ('prep', preprocessor), 
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
    ]
)

# random forest 
pipeline_rf = Pipeline(
    [
        ('prep', preprocessor), 
        ('clf', RandomForestClassifier(class_weight='balanced',  n_estimators=50, random_state=42))
    ]
)

#### **Splitting using naive approach(80-20)**

In [9]:
X = df.drop(columns = ['PitNextLap']) # features 
y = df['PitNextLap'] # target 
y = pd.DataFrame(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [13]:
# print("Random Forest")
# print(classification_report(y_test, y_pred_rf))
# print(roc_auc_score(y_test, y_prob_rf))

def evaluate(model, y_test, y_pred, y_prob):
    print("Random Forest")
    print(classification_report(y_test, y_pred_rf))
    display(roc_auc_score(y_test, y_prob_rf))


#### **Model Fit & Evaluation**

Decision Tree

In [10]:
pipeline_dt.fit(X_train, y_train)
y_pred_dt = pipeline_dt.predict(X_test)
y_prob_dt = pipeline_dt.predict_proba(X_test)[:,1]

Logisitic Regression

In [11]:
pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)
y_prob_lr = pipeline_lr.predict_proba(X_test)[:,1]

Random Forest

In [12]:
pipeline_rf.fit(X_train, y_train)
y_pred_rf = pipeline_rf.predict(X_test)
y_prob_rf = pipeline_rf.predict_proba(X_test)[:,1]